In [ ]:
# Imports
import copy,glob,json,os,random,re,time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu
import torch
import torch.nn as nn
from datasets import load_dataset
from gensim.models import FastText
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer,ENGLISH_STOP_WORDS as SW
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence,pad_packed_sequence
from torch.utils.data import DataLoader

In [2]:
# Settings: seed, where the Wikipedia documents are cloned, input/output length caps, vocabulary cut-off and model/training options
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DOG="datasets-CMU_DoG"
HLEN=60
DLEN=200
RLEN=30
MINF=2
BS=32
EPOCHS=30
PAT=4
LR=1e-3
D=128
H=256
DR=0.3
PAD,SOS,EOS,UNK=0,1,2,3
DEV="cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# Cleaning function for code-mixed text: lowercase, squeeze letters repeated 3+ times, urls to a word, split into word and punctuation tokens
def tok(s):
    s=re.sub(r"https?://\S+"," url ",str(s or "").lower())
    return " ".join(re.findall(r"\w+|[^\w\s]",re.sub(r"(.)\1{2,}",r"\1\1",s)))

In [4]:
# Load the utterance rows from Hugging Face and the 30 Wikipedia movie documents from GitHub (each document has 4 sections: 0 = intro, cast, reviews; 1-3 = key scenes)
ds=load_dataset("festvox/cmu_hinglish_dog")
raw={k:ds[k].to_pandas() for k in ds}
if not os.path.exists(DOG):os.system(f"git clone -q --depth 1 https://github.com/festvox/datasets-CMU_DoG {DOG}")
docs={}
for f in glob.glob(DOG+"/WikiData/*.json"):
    j=json.load(open(f))
    a=j["0"]
    docs[j["wikiDocumentIdx"]]=[tok(" ".join(map(str,[a["movieName"],a["year"],a["director"],a["genre"],a["introduction"],*a["rating"],*a["cast"],*a["critical_response"]])))]+[tok(j[str(i)]) for i in (1,2,3)]
ks=sorted(docs)
nx={k:ks[(i+1)%len(ks)] for i,k in enumerate(ks)}
print({k:len(v) for k,v in raw.items()},len(docs),"documents")

{'train': 8060, 'test': 960, 'validation': 942} 30 documents


In [5]:
# Rows are single utterances, so rebuild conversations (a new one starts when document id, user2 id or login times change) and make one example per reply: speaker-tagged history (<me>/<you>), the reply, the document section being read, whether the speaker saw the document, and how code-mixed the reply is
def cmr(a,b):
    x=[w for w in a.split() if w.isalpha()]
    return np.mean([w not in set(b.split()) for w in x]) if x else 0.0
def build(d):
    d=d.reset_index(drop=True)
    k=d[["wikiDocumentIdx","user2_id","uid1LogInTime","uid1LogOutTime"]].fillna("").astype(str).agg("|".join,axis=1)
    rows=[]
    for _,g in d.groupby((k!=k.shift()).cumsum()):
        hist=[]
        for r in g.itertuples():
            u=tok(r.translation["hi_en"])
            e=tok(r.translation["en"])
            if not u:continue
            if hist:
                h=[w for p,t in hist for w in ["<me>" if p==r.uid else "<you>"]+t.split()][-HLEN:]
                rows.append({"h":" ".join(h),"q":hist[-1][1],"w":r.wikiDocumentIdx,"s":r.docIdx,"r":" ".join(u.split()[:RLEN]),"cm":cmr(u,e),"saw":r.uid in list(r.whoSawDoc)})
            hist.append((r.uid,u))
    return pd.DataFrame(rows)
tr,va,te=build(raw["train"]),build(raw["validation"]),build(raw["test"])
te["cmb"]=pd.cut(te.cm,[-.01,.2,.5,1.01],labels=["mostly English","mixed","heavily Hinglish"])
print(len(tr),len(va),len(te),"| replies by a speaker who saw the document:",round(tr.saw.mean(),3))
print(tr.head(3).to_string())

7803 910 928 | replies by a speaker who saw the document: 0.773
                                                                                                                                       h                                                                                           q   w  s                                                                                           r        cm    saw
0                                                                                                                                <me> hi                                                                                          hi  14  0                                                                     tumne konsi movie dekhi  1.000000  False
1                                                                                                 <you> hi <you> tumne konsi movie dekhi                                                                     tumne konsi movie dekhi  14  0  hello tum

In [6]:
# Embeddings from scratch: train subword (char n-gram) skip-gram FastText on the training utterances and the documents, so spelling variants of romanized Hindi share vectors; one vocabulary and one embedding table are shared by history, document and reply
U=[tok(t["hi_en"]) for t in raw["train"].translation]
S=[u.split() for u in U]
for v in docs.values():
    for t in v:S+=[x.split() for x in t.split(" . ")]
c=Counter(w for s in S for w in s)
i2w=["<pad>","<sos>","<eos>","<unk>","<me>","<you>"]+[w for w,n in c.items() if n>=MINF]
w2i={w:i for i,w in enumerate(i2w)}
ft=FastText(S,vector_size=D,window=5,min_count=1,sg=1,min_n=2,max_n=5,epochs=30,seed=SEED,workers=1)
E=np.stack([ft.wv[w] for w in i2w]).astype(np.float32)
E[PAD]=0
print("vocab",len(i2w),"| similar to 'movie':",[w for w,_ in ft.wv.most_similar("movie",topn=5)])

vocab 6279 | similar to 'movie': ['moovie', 'movi', 'movbie', 'movies', 'maovie']


In [7]:
# Convert to ids and build padded mini-batches (history, document, decoder input with <sos>, decoder target with <eos>); sh=True swaps in the document of a different movie
ids=lambda s:[w2i.get(x,UNK) for x in s.split()]
def pairs(d,sh=False):
    return [(ids(h),ids(docs[nx[w] if sh else w][s])[:DLEN],ids(r)) for h,w,s,r in zip(d.h,d.w,d.s,d.r)]
def coll(b):
    h,d,r=zip(*b)
    f=lambda s:pad_sequence([torch.tensor(i) for i in s],batch_first=True)
    return f(h),torch.tensor([len(i) for i in h]),f(d),torch.tensor([len(i) for i in d]),f([[SOS]+i for i in r]),f([i+[EOS] for i in r])
trl=DataLoader(pairs(tr),batch_size=BS,shuffle=True,collate_fn=coll)
val=DataLoader(pairs(va),batch_size=BS,collate_fn=coll)
tel=DataLoader(pairs(te),batch_size=BS,collate_fn=coll)
tew=DataLoader(pairs(te,True),batch_size=BS,collate_fn=coll)

In [8]:
# Model: Exp 2 encoder-decoder with Luong attention, extended to two sources; one LSTM encodes the history, a second encodes the document, and at every step the decoder attends over both and combines the two contexts with its own state (doc=False gives the ungrounded baseline)
class Net(nn.Module):
    def __init__(self,V,doc):
        super().__init__()
        self.doc=doc
        self.emb=nn.Embedding.from_pretrained(torch.tensor(E),freeze=False,padding_idx=PAD)
        self.he=nn.LSTM(D,H,batch_first=True)
        self.dec=nn.LSTM(D,H,batch_first=True)
        self.Wh=nn.Linear(H,H,bias=False)
        if doc:
            self.de=nn.LSTM(D,H,batch_first=True)
            self.Wd=nn.Linear(H,H,bias=False)
        self.C=nn.Linear(H*(3 if doc else 2),H)
        self.out=nn.Linear(H,V)
        self.dr=nn.Dropout(DR)
    def enc(self,lstm,x,l):
        o,st=lstm(pack_padded_sequence(self.dr(self.emb(x)),l.cpu(),batch_first=True,enforce_sorted=False))
        return pad_packed_sequence(o,batch_first=True,total_length=x.size(1))[0],st
    def att(self,W,o,e,mk):
        return torch.bmm(torch.softmax(torch.bmm(W(o),e.transpose(1,2)).masked_fill(mk[:,None,:],-1e9),-1),e)
    def memo(self,h,hl,d,dl):
        eh,st=self.enc(self.he,h,hl)
        return st,eh,h==PAD,(self.enc(self.de,d,dl)[0] if self.doc else None),d==PAD
    def decode(self,x,st,eh,mh,ed,md):
        o,st=self.dec(self.dr(self.emb(x)),st)
        c=[o,self.att(self.Wh,o,eh,mh)]
        if self.doc:c.append(self.att(self.Wd,o,ed,md))
        return self.out(self.dr(torch.tanh(self.C(torch.cat(c,-1))))),st
    def forward(self,h,hl,d,dl,x):
        st,*m=self.memo(h,hl,d,dl)
        return self.decode(x,st,*m)[0]

In [ ]:
# Training for both models: teacher forcing, cross-entropy, Adam, gradient clipping, early stopping on validation loss
lf=nn.CrossEntropyLoss(ignore_index=PAD)
def ep(m,l,opt=None):
    m.train(opt is not None)
    s=n=0
    with torch.set_grad_enabled(opt is not None):
        for h,hl,d,dl,X,Y in l:
            h,d,X,Y=h.to(DEV),d.to(DEV),X.to(DEV),Y.to(DEV)
            o=m(h,hl,d,dl,X)
            loss=lf(o.reshape(-1,o.size(-1)),Y.reshape(-1))
            if opt is not None:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(),1.0)
                opt.step()
            k=(Y!=PAD).sum().item()
            s+=loss.item()*k
            n+=k
    return s/n
def fit(doc):
    torch.manual_seed(SEED)
    m=Net(len(i2w),doc).to(DEV)
    opt=torch.optim.Adam(m.parameters(),lr=LR)
    hs={"train":[],"val":[]}
    bv,st,bad=1e9,None,0
    t0=time.time()
    for e in range(1,EPOCHS+1):
        hs["train"].append(ep(m,trl,opt))
        hs["val"].append(ep(m,val))
        print(f"{e:02d} train {hs['train'][-1]:.4f} val {hs['val'][-1]:.4f} val ppl {np.exp(hs['val'][-1]):.2f}")
        if hs["val"][-1]<bv:bv,st,bad=hs["val"][-1],copy.deepcopy(m.state_dict()),0
        else:
            bad+=1
            if bad>=PAT:break
    m.load_state_dict(st)
    print("params",sum(p.numel() for p in m.parameters()),"| seconds",round(time.time()-t0))
    return m,hs
M={}
for k,doc in [("Ungrounded",False),("Grounded",True)]:
    print("\n"+k)
    M[k]=fit(doc)


Ungrounded
01 train 6.3591 val 5.6461 val ppl 283.19
02 train 5.6055 val 5.1381 val ppl 170.39
03 train 5.1513 val 4.8764 val ppl 131.15
04 train 4.8510 val 4.7508 val ppl 115.67
05 train 4.6288 val 4.6633 val ppl 105.98
06 train 4.4478 val 4.6057 val ppl 100.06
07 train 4.2862 val 4.5870 val ppl 98.20
08 train 4.1354 val 4.5739 val ppl 96.92
09 train 3.9944 val 4.5739 val ppl 96.92


In [ ]:
# Training vs validation loss for both models
for i,k in enumerate(M):
    plt.plot(M[k][1]["train"],color=f"C{i}",label=k+" train")
    plt.plot(M[k][1]["val"],"--",color=f"C{i}",label=k+" validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy per token")
plt.legend()
plt.show()

In [ ]:
# Greedy generation, plus the retrieval baseline (reply of the training example whose last turn is most similar by TF-IDF) and the metric functions: BLEU, ROUGE, distinct-n (variety of words), and document overlap (share of content words of a reply that appear in the true document section)
def gen(m,l):
    m.eval()
    out=[]
    with torch.no_grad():
        for h,hl,d,dl,_,_ in l:
            h,d=h.to(DEV),d.to(DEV)
            st,*mm=m.memo(h,hl,d,dl)
            x=torch.full((len(h),1),SOS,dtype=torch.long,device=DEV)
            o=[]
            for _ in range(RLEN+1):
                y,st=m.decode(x,st,*mm)
                x=y[:,-1].argmax(-1,keepdim=True)
                o.append(x)
            for r in torch.cat(o,1).tolist():
                out.append(" ".join(i2w[i] for i in (r[:r.index(EOS)] if EOS in r else r)))
    return np.array(out,dtype=object)
vz=TfidfVectorizer(token_pattern=r"\S+").fit(tr.q)
ret=lambda d:tr.r.to_numpy()[np.asarray((vz.transform(d.q)@vz.transform(tr.q).T).argmax(1)).ravel()]
bl=lambda h,r:sacrebleu.corpus_bleu(list(h),[list(r)],tokenize="none",force=True).score
rs=rouge_scorer.RougeScorer(["rouge1","rougeL"])
def dist(Hy,n):
    g=[tuple(w[i:i+n]) for w in (h.split() for h in Hy) for i in range(len(w)-n+1)]
    return len(set(g))/max(len(g),1)
def dov(h,d):
    x=[w for w in h.split() if w.isalpha() and len(w)>3 and w not in SW]
    return np.mean([w in set(d.split()) for w in x]) if x else np.nan
TD=[docs[w][s] for w,s in zip(te.w,te.s)]
def ev(Hy,Rf):
    s=[rs.score(b,a) for a,b in zip(Hy,Rf)]
    return {"BLEU":bl(Hy,Rf),"ROUGE-1":np.mean([x["rouge1"].fmeasure for x in s]),"ROUGE-L":np.mean([x["rougeL"].fmeasure for x in s]),"Dist-1":dist(Hy,1),"Dist-2":dist(Hy,2),"AvgLen":np.mean([len(h.split()) for h in Hy]),"DocOverlap":np.nanmean([dov(h,d) for h,d in zip(Hy,TD)])}
Rf=te.r.to_numpy()
HY={"Retrieval":ret(te),"Ungrounded":gen(M["Ungrounded"][0],tel),"Grounded":gen(M["Grounded"][0],tel),"Grounded (wrong movie's document)":gen(M["Grounded"][0],tew)}
tab=pd.DataFrame({k:ev(v,Rf) for k,v in HY.items()}).T
tab.loc["Gold reference"]=ev(Rf,Rf)
tab.loc["Gold reference",["BLEU","ROUGE-1","ROUGE-L"]]=np.nan
print(tab.round(3).to_string())

In [ ]:
# Does the model use the document, and where does it break: scores split by whether the speaker had seen the document, and by how code-mixed the reference reply is
def by(c):
    r={}
    for b,g in te.groupby(c,observed=True):
        r[b]={"n":len(g)}
        for k in ["Ungrounded","Grounded"]:
            r[b][k+" BLEU"]=bl(HY[k][g.index],g.r)
            r[b][k+" ROUGE-L"]=ev(HY[k][g.index],g.r.to_numpy())["ROUGE-L"]
    return pd.DataFrame(r).T.round(3)
print("By whether the speaker saw the document:")
print(by("saw").to_string())
print("\nBy code-mixing of the reference reply:")
print(by("cmb").to_string())

In [ ]:
# Example replies: 6 random test turns where the speaker saw the document, with the history, the reference and every system's reply
for i in np.random.RandomState(SEED).choice(np.where(te.saw)[0],6,replace=False):
    print("HISTORY:"," ".join(te.h[i].split()[-30:]),"\nDOC    :"," ".join(TD[i].split()[:25]),"...\nREF    :",Rf[i])
    for k in HY:print(f"{k}:",HY[k][i])
    print()